# Calibration-Efficient, Uncertainty-Aware Gradient Boosting for EEG Mental-Workload Detection

**Just run top to bottom: \`Runtime -> Run all\`.**

Pipeline: STEW (Emotiv 14ch) -> filter -> 4 s windows -> theta/alpha/beta band power + ratios + frontal asymmetry -> XGBoost / LightGBM / CatBoost vs SVM / RF -> 10-fold AND leave-one-subject-out (LOSO) -> latency + Friedman/Nemenyi + SHAP -> **calibration-efficiency** + **conformal selective prediction** -> auto-built **.docx paper** with figures and tables.

**Before running:** put your \`STEW Dataset.zip\` in your Google Drive (My Drive). Cell 5 mounts Drive and finds it automatically. Get STEW from IEEE DataPort: https://ieee-dataport.org/open-access/stew-simultaneous-task-eeg-workload-dataset

## 1. Install dependencies

In [ ]:
!pip install -q xgboost lightgbm catboost shap scikit-posthocs 2>/dev/null
print('Dependencies installed.')

## 2. Imports & configuration

In [ ]:
import os, glob, time, warnings
import numpy as np, pandas as pd
from scipy import signal as sps
from scipy.stats import friedmanchisquare
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score, roc_auc_score)
import xgboost as xgb, lightgbm as lgb
from catboost import CatBoostClassifier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

RNG = 42
np.random.seed(RNG)

FS_STEW = 128
WIN_SEC = 4.0
BANDS = {'theta': (4, 8), 'alpha': (8, 13), 'beta': (13, 30)}
EPOC_CH = ['AF3','F7','F3','FC5','T7','P7','O1','O2','P8','T8','FC6','F4','F8','AF4']
ASYM_PAIRS = [('AF3','AF4'),('F7','F8'),('F3','F4'),('FC5','FC6'),('T7','T8'),('P7','P8'),('O1','O2')]
ARTIFACT_UV = 500.0   # Emotiv blinks are large; 500 keeps all 48 subjects
print('Config ready.')

## 3. Signal processing & feature extraction

In [ ]:
def preprocess(x, fs):
    b, a = sps.butter(4, [1, 40], btype='band', fs=fs)
    x = sps.filtfilt(b, a, x, axis=-1)
    for f0 in (50, 60):
        if f0 < fs/2:
            bn, an = sps.iirnotch(f0, 30, fs); x = sps.filtfilt(bn, an, x, axis=-1)
    return x

def windowize(x, fs, win_sec=WIN_SEC):
    w = int(win_sec*fs); n = x.shape[1]//w
    return np.stack([x[:, i*w:(i+1)*w] for i in range(n)]) if n else np.empty((0,))

def bandpowers(win, fs, ch_names):
    freqs, psd = sps.welch(win, fs=fs, nperseg=min(win.shape[1], int(fs*2)), axis=-1)
    feats, names = [], []
    total = np.trapz(psd, freqs, axis=-1) + 1e-12; bp = {}
    for bname,(lo,hi) in BANDS.items():
        idx = (freqs>=lo)&(freqs<hi)
        p = np.trapz(psd[:, idx], freqs[idx], axis=-1); bp[bname] = p; rel = p/total
        for c in range(len(ch_names)):
            feats += [p[c], rel[c]]; names += [f'{bname}_abs_{ch_names[c]}', f'{bname}_rel_{ch_names[c]}']
    tb = bp['theta']/(bp['beta']+1e-12); ab = bp['alpha']/(bp['beta']+1e-12)
    for c in range(len(ch_names)):
        feats += [tb[c], ab[c]]; names += [f'theta_beta_{ch_names[c]}', f'alpha_beta_{ch_names[c]}']
    cidx = {c:i for i,c in enumerate(ch_names)}
    for L,R in ASYM_PAIRS:
        if L in cidx and R in cidx:
            feats.append(np.log(bp['alpha'][cidx[R]]+1e-12)-np.log(bp['alpha'][cidx[L]]+1e-12))
            names.append(f'alpha_asym_{L}_{R}')
    return np.array(feats), names

def extract_subject(x, fs, label, ch_names):
    x = preprocess(x, fs); wins = windowize(x, fs)
    if wins.size == 0: return None, None, None
    X, names = [], None
    for w in wins:
        if np.any((w.max(-1)-w.min(-1)) > ARTIFACT_UV): continue
        f, names = bandpowers(w, fs, ch_names); X.append(f)
    if not X: return None, None, None
    return np.vstack(X), np.full(len(X), label), names

def load_stew(root='./data/STEW'):
    Xs, ys, groups, names = [], [], [], None
    files = sorted(glob.glob(os.path.join(root, '*.txt')))
    if not files: raise FileNotFoundError(f'No STEW .txt in {root}')
    for fp in files:
        base = os.path.basename(fp).lower(); lbl = 1 if 'hi' in base else 0
        raw = np.loadtxt(fp).T
        if raw.shape[0] != 14: raw = raw[:14]
        X, y, names = extract_subject(raw, FS_STEW, lbl, EPOC_CH)
        if X is None: continue
        Xs.append(X); ys.append(y); groups += [base.split('_')[0]]*len(y)
    return np.vstack(Xs), np.concatenate(ys), np.array(groups), names

print('Feature + loader functions ready.')

## 4. Load STEW from Google Drive
Put **STEW Dataset.zip** anywhere in your Google Drive (My Drive). This mounts Drive, finds the zip, unzips inside Colab, and loads all 96 files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile, shutil
ZIP = glob.glob('/content/drive/MyDrive/**/*STEW*.zip', recursive=True)
assert ZIP, "Put a file named like 'STEW Dataset.zip' in your Google Drive (My Drive)."
print('Found:', ZIP[0])
with zipfile.ZipFile(ZIP[0]) as z: z.extractall('/content/stew_raw')
os.makedirs('./data/STEW', exist_ok=True)
for p in glob.glob('/content/stew_raw/**/*.txt', recursive=True):
    if 'ratings' not in os.path.basename(p).lower():
        shutil.copy(p, os.path.join('./data/STEW', os.path.basename(p)))
print(len(glob.glob('./data/STEW/*.txt')), 'recording files ready')
X, y, groups, names = load_stew()
print(f'{X.shape[0]} windows, {X.shape[1]} features, {len(set(groups))} subjects, classes={np.bincount(y)}')

## 5. Models & evaluation

In [ ]:
def make_models():
    return {
        'XGBoost': xgb.XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, eval_metric='logloss',
            random_state=RNG, n_jobs=-1),
        'LightGBM': lgb.LGBMClassifier(n_estimators=400, num_leaves=31, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, random_state=RNG, n_jobs=-1, verbose=-1),
        'CatBoost': CatBoostClassifier(iterations=400, depth=5, learning_rate=0.05,
            l2_leaf_reg=3.0, random_seed=RNG, verbose=0),
        'SVM_RBF': SVC(C=10, gamma='scale', probability=True, random_state=RNG),
        'RandomForest': RandomForestClassifier(n_estimators=400, random_state=RNG, n_jobs=-1),
    }

def evaluate(model, Xtr, ytr, Xte, yte):
    sc = StandardScaler().fit(Xtr)               # fit on TRAIN only -> no leakage
    Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    model.fit(Xtr, ytr)
    t0 = time.perf_counter(); pred = model.predict(Xte)
    lat = (time.perf_counter()-t0)/max(len(Xte),1)*1000
    try: auc = roc_auc_score(yte, model.predict_proba(Xte)[:,1])
    except Exception: auc = np.nan
    return dict(acc=accuracy_score(yte,pred), f1=f1_score(yte,pred,average='macro'),
                kappa=cohen_kappa_score(yte,pred), auc=auc, latency_ms=lat)

def run_protocol(X, y, groups, splitter, split_args):
    res = {m: [] for m in make_models()}
    for tr, te in splitter.split(X, y, *split_args):
        for name, model in make_models().items():
            res[name].append(evaluate(model, X[tr], y[tr], X[te], y[te]))
    return res

def summarize(res, title):
    print(f'\n=== {title} ===')
    rows = []
    for m, folds in res.items():
        rows.append([m,
            np.nanmean([f['acc'] for f in folds]), np.nanmean([f['f1'] for f in folds]),
            np.nanmean([f['kappa'] for f in folds]), np.nanmean([f['auc'] for f in folds]),
            np.nanmean([f['latency_ms'] for f in folds])])
    df = pd.DataFrame(rows, columns=['Model','Acc','F1','Kappa','AUC','ms/win']).round(3)
    display(df)
    return df, {m:[f['acc'] for f in res[m]] for m in res}
print('Eval functions ready.')

## 6. Within-subject (10-fold) -> Table 1

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RNG)
df_10f, _ = summarize(run_protocol(X, y, groups, skf, ()), 'Within-subject (10-fold)')

## 7. Leave-One-Subject-Out (PRIMARY) -> Table 2

In [ ]:
logo = LeaveOneGroupOut()
res_loso = run_protocol(X, y, groups, logo, (groups,))
df_loso, acc_table = summarize(res_loso, 'Leave-One-Subject-Out (LOSO)')

## 8. Statistics (Friedman + Nemenyi)

In [ ]:
import scikit_posthocs as sp
df_acc = pd.DataFrame(acc_table)
stat, p = friedmanchisquare(*[df_acc[c].values for c in df_acc.columns])
print(f'Friedman chi2={stat:.3f}, p={p:.4g}')
if p < 0.05:
    print('Nemenyi post-hoc p-values:')
    display(sp.posthoc_nemenyi_friedman(df_acc.values).round(4).set_axis(df_acc.columns).set_axis(df_acc.columns, axis=1))

## 10. Build the complete paper (.docx) with figures + tables
Self-contained: computes SHAP, calibration-efficiency, conformal selective prediction, renders 3 figures, embeds everything, and downloads **EEG_paper_COMPLETE.docx**.

## 9. (Optional) Cross-corpus transfer probe on DREAMER
Run this BEFORE the paper cell if you want the cross-corpus result in the paper. Paste a Kaggle slug OR a direct URL below; it downloads into Colab disk (no Drive needed) and also checks Drive. Skips cleanly if nothing is found. Exploratory: DREAMER labels arousal, not workload.

In [ ]:
dreamer_acc = None  # default: paper omits the section unless this probe succeeds
import glob, zipfile
SLUG = ""   # Kaggle dataset slug, e.g. "username/dreamer-dataset" (after kaggle.com/datasets/)
URL  = ""   # OR a direct download link to a .mat or .zip

os.makedirs('/content/dreamer', exist_ok=True)
if SLUG:
    !pip -q install kaggle
    from google.colab import files
    print("Upload kaggle.json (Kaggle -> Settings -> Create New API Token):"); files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d "$SLUG" -p /content/dreamer --unzip
elif URL:
    !wget -q -O /content/dreamer/dl "$URL"
    if zipfile.is_zipfile('/content/dreamer/dl'):
        with zipfile.ZipFile('/content/dreamer/dl') as z: z.extractall('/content/dreamer')
    else:
        os.rename('/content/dreamer/dl', '/content/dreamer/DREAMER.mat')

mats = glob.glob('/content/dreamer/**/*.mat', recursive=True) + glob.glob('/content/drive/MyDrive/**/DREAMER*.mat', recursive=True)
print("MAT files:", mats)
if mats:
    from scipy.io import loadmat
    try:
        D = loadmat(mats[0])['DREAMER'][0,0]; subs = D['Data'][0]
        Xd, yd = [], []
        for s in range(len(subs)):
            rec = subs[s][0,0]; eeg = rec['EEG'][0,0]; stim = eeg['stimuli'][:,0]
            ar = np.asarray(rec['ScoreArousal']).ravel().astype(float); thr = np.median(ar)
            for t in range(len(stim)):
                sig = np.asarray(stim[t]).T
                if sig.shape[0] != 14: continue
                Xe,_,_ = extract_subject(sig, 128, int(ar[t] > thr), EPOC_CH)
                if Xe is None: continue
                Xd.append(Xe); yd += [int(ar[t] > thr)]*len(Xe)
        Xd = np.vstack(Xd); yd = np.array(yd)
        sc = StandardScaler().fit(X)
        m = xgb.XGBClassifier(n_estimators=400,max_depth=5,learning_rate=0.05,subsample=0.8,
            colsample_bytree=0.8,reg_lambda=1.0,eval_metric='logloss',random_state=RNG,n_jobs=-1).fit(sc.transform(X), y)
        dreamer_acc = float((m.predict(sc.transform(Xd)) == yd).mean())
        dreamer_n = int(len(yd)); dreamer_maj = float(max(yd.mean(), 1-yd.mean()))
        print(f"Cross-corpus STEW->DREAMER: n={dreamer_n}, acc={dreamer_acc:.3f}, majority baseline={dreamer_maj:.3f}")
        print("Different construct (workload vs arousal): exploratory transfer probe, not a benchmark.")
    except Exception as e:
        print("MAT parse failed (struct varies by release):", repr(e))
        print("Inspect with: loadmat(mats[0])['DREAMER'][0,0].dtype.names ; then tell me the fields.")
else:
    print("No .mat found -> paper will omit the cross-corpus section (this is fine).")

In [ ]:
# ================================================================
#  Builds the complete paper (.docx) with figures + tables.
#  Computes SHAP + calibration-efficiency + conformal internally.
# ================================================================
!apt-get -qq install -y pandoc >/dev/null 2>&1
import os, numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
import xgboost as xgb, lightgbm as lgb
from catboost import CatBoostClassifier
import shap

RNG=42
NICE={'XGBoost':'XGBoost','LightGBM':'LightGBM','CatBoost':'CatBoost','SVM_RBF':'SVM (RBF)','RandomForest':'Random Forest'}
BOOST=['XGBoost','LightGBM','CatBoost']
def look(df,m,c):
    try: return float(df.loc[df['Model']==m,c].values[0])
    except Exception: return None
def gmd(df):
    return "\n".join(["| "+" | ".join(map(str,df.columns))+" |",
                      "|"+"|".join(["---"]*len(df.columns))+"|"]+
                     ["| "+" | ".join(str(v) for v in r)+" |" for r in df.values])
def md_models(df):
    d=df.copy(); d['Model']=d['Model'].map(lambda m:NICE.get(m,m)); return gmd(d)
def f3(m,df): v=look(df,m,'Acc'); return f"{v:.3f}" if v is not None else "n/a"

bi=df_loso['Acc'].astype(float).idxmax(); best=df_loso.loc[bi,'Model']
bla=look(df_loso,best,'Acc'); blf=look(df_loso,best,'F1'); bll=look(df_loso,best,'ms/win')
b10=look(df_10f,best,'Acc') or bla; gap=round((b10-bla)*100,1)
svm=look(df_loso,'SVM_RBF','Acc') or bla; rf=look(df_loso,'RandomForest','Acc') or bla
margin=round((bla-svm)*100,1)
bl=df_loso[df_loso['Model'].isin(BOOST)]['ms/win'].astype(float)
xl=df_loso[~df_loso['Model'].isin(BOOST)]['ms/win'].astype(float)
blmax=round(float(bl.max()),2); xlo=round(float(xl.min()),2); xhi=round(float(xl.max()),2)
dfa=pd.DataFrame(acc_table); fc,fp=friedmanchisquare(*[dfa[c].values for c in dfa.columns])
fdf=dfa.shape[1]-1; fps="< 0.001" if fp<0.001 else f"= {fp:.3f}"

def boost():
    if best=='LightGBM': return lgb.LGBMClassifier(n_estimators=400,num_leaves=31,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,reg_lambda=1.0,random_state=RNG,n_jobs=-1,verbose=-1)
    if best=='CatBoost': return CatBoostClassifier(iterations=400,depth=5,learning_rate=0.05,l2_leaf_reg=3.0,random_seed=RNG,verbose=0)
    return xgb.XGBClassifier(n_estimators=400,max_depth=5,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,reg_lambda=1.0,eval_metric='logloss',random_state=RNG,n_jobs=-1)

print("SHAP..."); sc0=StandardScaler().fit(X)
sm=boost().fit(sc0.transform(X),y); sv=shap.TreeExplainer(sm).shap_values(sc0.transform(X))
sv=sv[1] if isinstance(sv,list) else sv; imp=np.abs(sv).mean(0); order=np.argsort(imp)[::-1]
top3=", ".join(names[i] for i in order[:3])
idx=order[:15]; plt.figure(figsize=(7,5))
plt.barh([names[i] for i in idx][::-1], imp[idx][::-1]); plt.xlabel('mean |SHAP value|')
plt.title(f'Top 15 SHAP features ({NICE[best]})'); plt.tight_layout(); plt.savefig('fig_shap.png',dpi=150); plt.close()

print("Calibration sweep..."); logo=LeaveOneGroupOut()
def calibration(ks=(0,5,10,20,40)):
    rng=np.random.RandomState(RNG); rows=[]
    for k in ks:
        accs=[]
        for tr,te in logo.split(X,y,groups):
            Xtr,ytr=X[tr].copy(),y[tr].copy(); Xte,yte=X[te],y[te]; ti=np.arange(len(te))
            if k>0:
                cal=[]
                for c in np.unique(yte):
                    ci=ti[yte==c]; rng.shuffle(ci)
                    take=min(max(1,k//2), max(0,len(ci)-1))
                    cal+=list(ci[:take])
                cal=np.array(cal,dtype=int); keep=np.setdiff1d(ti,cal)
                if len(keep)==0 or len(cal)==0: continue
                Xtr=np.vstack([Xtr,Xte[cal]]); ytr=np.concatenate([ytr,yte[cal]]); Xt,yt=Xte[keep],yte[keep]
            else: Xt,yt=Xte,yte
            scc=StandardScaler().fit(Xtr); m=boost().fit(scc.transform(Xtr),ytr)
            accs.append((m.predict(scc.transform(Xt))==yt).mean())
        rows.append([k,k*4,round(np.mean(accs),3),round(np.std(accs),3)])
        print(f"  k={k}: {np.mean(accs):.3f}")
    return pd.DataFrame(rows,columns=['Calib. windows','Seconds','Accuracy','SD'])
cal=calibration()
cal_k0=float(cal.loc[cal['Calib. windows']==0,'Accuracy']); cal_k20=float(cal.loc[cal['Calib. windows']==20,'Accuracy'])
cal_gain=round((cal_k20-cal_k0)*100,1)
plt.figure(figsize=(6,4)); plt.errorbar(cal['Calib. windows'],cal['Accuracy'],yerr=cal['SD'],marker='o',capsize=3)
plt.xlabel('calibration windows per new user'); plt.ylabel('LOSO accuracy'); plt.title('Calibration efficiency')
plt.grid(alpha=.3); plt.tight_layout(); plt.savefig('fig_calibration.png',dpi=150); plt.close()

print("Conformal...")
def conformal(alpha):
    rng=np.random.RandomState(RNG); cov,sg,sa=[],[],[]
    for tr,te in logo.split(X,y,groups):
        ii=np.arange(len(tr)); rng.shuffle(ii); cut=int(.8*len(ii)); ptr,cx=tr[ii[:cut]],tr[ii[cut:]]
        scc=StandardScaler().fit(X[ptr]); m=boost().fit(scc.transform(X[ptr]),y[ptr])
        pc=m.predict_proba(scc.transform(X[cx])); s=1-pc[np.arange(len(cx)),y[cx]]
        n=len(s); ql=min(1.0,np.ceil((n+1)*(1-alpha))/n); qh=np.quantile(s,ql,method='higher')
        pt=m.predict_proba(scc.transform(X[te])); sets=pt>=(1-qh)
        cov.append(sets[np.arange(len(te)),y[te]].mean()); single=sets.sum(1)==1; sg.append(single.mean())
        if single.sum(): sa.append((np.argmax(pt,1)[single]==y[te][single]).mean())
    return round(np.mean(cov),3),round(np.mean(sg),3),round(np.nanmean(sa),3)
conf_rows=[]
for a in (0.20,0.10,0.05):
    c,s,ac=conformal(a); conf_rows.append([f"{1-a:.2f}",c,s,ac]); print(f"  cov_target={1-a:.2f}: cov={c} conf={s} sel_acc={ac}")
conf=pd.DataFrame(conf_rows,columns=['Target cov.','Empirical cov.','Confident frac.','Selective acc.'])
c10=conf.iloc[1]

print("Ablations: bootstrap CIs + channel montages...")
from numpy.random import default_rng
def boot_ci(vals,n=3000,seed=42):
    rng=default_rng(seed); vals=np.array(vals)
    return np.percentile([rng.choice(vals,len(vals),replace=True).mean() for _ in range(n)],[2.5,97.5])
ci_lo,ci_hi=boot_ci(acc_table[best])
def feats_for(chans):
    cs=set(chans); keep=[]
    for i,nm in enumerate(names):
        if nm.startswith('alpha_asym_'):
            pr=nm.split('_')
            if pr[2] in cs and pr[3] in cs: keep.append(i)
        elif nm.split('_')[-1] in cs: keep.append(i)
    return np.array(keep)
def loso_sub(cols):
    a=[]
    for tr,te in logo.split(X,y,groups):
        scc=StandardScaler().fit(X[tr][:,cols]); m=boost().fit(scc.transform(X[tr][:,cols]),y[tr])
        a.append((m.predict(scc.transform(X[te][:,cols]))==y[te]).mean())
    return round(np.mean(a),3)
montages=[('14 (full Emotiv)',EPOC_CH),('8 fronto-temporal',['AF3','AF4','F7','F8','F3','F4','T7','T8']),
          ('4 (Muse-like)',['AF3','AF4','T7','T8']),('2 (frontal)',['AF3','AF4'])]
mrows=[]
for nm,ch in montages:
    cols=feats_for(ch); acc=loso_sub(cols); mrows.append([nm,len(ch),len(cols),acc]); print(f"  {nm}: {acc}")
mont=pd.DataFrame(mrows,columns=['Montage','Channels','Features','LOSO acc'])
mont4=float(mont.loc[mont['Channels']==4,'LOSO acc'].values[0]); mont14=float(mont.loc[mont['Channels']==14,'LOSO acc'].values[0])

plt.figure(figsize=(11,4))
a1=plt.subplot(1,2,1); xp=np.arange(len(df_10f)); w=.38
a1.bar(xp-w/2,df_10f['Acc'],w,label='10-fold'); a1.bar(xp+w/2,df_loso['Acc'],w,label='LOSO')
a1.set_xticks(xp); a1.set_xticklabels([NICE.get(m,m) for m in df_10f['Model']],rotation=45,ha='right')
a1.set_ylabel('Accuracy'); a1.set_ylim(.5,1); a1.legend(); a1.set_title('Within-subject vs LOSO')
a2=plt.subplot(1,2,2); a2.scatter(df_loso['ms/win'],df_loso['Acc'])
for _,r in df_loso.iterrows(): a2.annotate(NICE.get(r['Model'],r['Model']),(r['ms/win'],r['Acc']),fontsize=8)
a2.set_xlabel('Inference (ms/window)'); a2.set_ylabel('LOSO accuracy'); a2.set_title('Accuracy vs latency')
plt.tight_layout(); plt.savefig('fig_acc_latency.png',dpi=150); plt.close()

# ---- optional DREAMER cross-corpus result (filled only if the probe ran) ----
dreamer_acc = globals().get('dreamer_acc', None)
_dmaj = float(globals().get('dreamer_maj', 0.5))
if dreamer_acc and dreamer_acc > _dmaj + 0.02:   # only report if it actually beats the majority baseline
    dn = int(globals().get('dreamer_n', 0)); dmaj = _dmaj
    da = f"{dreamer_acc*100:.1f}"; dmajs = f"{dmaj*100:.1f}"
    DREAMER_SECTION = ("### 4.8 Cross-corpus transfer (exploratory)\n"
        f"To probe transfer beyond a single corpus, the STEW-trained model was applied without retraining to DREAMER, an independent Emotiv recording labelled for emotional arousal rather than workload. On {dn} windows it reached {da}% accuracy against a majority-class baseline of {dmajs}%. The constructs differ, so this is not a like-for-like benchmark, but performance above the baseline indicates the frontal band-power representation carries information that is not purely workload-specific and survives a change of cohort, session and task.\n")
    DREAMER_ABS = f" A model trained on workload and applied unchanged to a second Emotiv corpus labelled for arousal transferred above its majority-class baseline ({da}% versus {dmajs}%), hinting the representation is not narrowly task-specific."
else:
    DREAMER_SECTION = ""; DREAMER_ABS = ""

SUB={"{{BEST}}":NICE.get(best,best),"{{PCT}}":f"{bla*100:.1f}","{{DEC}}":f"{bla:.3f}","{{F1}}":f"{blf:.3f}",
"{{LAT}}":f"{bll:.2f}","{{GAP}}":f"{gap:g}","{{MARGIN}}":f"{margin:g}","{{SVM}}":f"{svm:.3f}","{{RF}}":f"{rf:.3f}",
"{{XGB10}}":f3('XGBoost',df_10f),"{{CAT10}}":f3('CatBoost',df_10f),"{{BLAT}}":f"{blmax:g}","{{XRANGE}}":f"{xlo:g} to {xhi:g}",
"{{FDF}}":str(fdf),"{{CHI2}}":f"{fc:.2f}","{{FP}}":fps,"{{TOP3}}":top3,
"{{T1}}":md_models(df_10f),"{{T2}}":md_models(df_loso),"{{CALT}}":gmd(cal),"{{CONFT}}":gmd(conf),
"{{CK0}}":f"{cal_k0*100:.1f}","{{CK20}}":f"{cal_k20*100:.1f}","{{CGAIN}}":f"{cal_gain:g}",
"{{COV}}":f"{float(c10['Empirical cov.'])*100:.1f}","{{CFRAC}}":f"{float(c10['Confident frac.'])*100:.1f}","{{CSEL}}":f"{float(c10['Selective acc.'])*100:.1f}",
"{{CILO}}":f"{ci_lo:.3f}","{{CIHI}}":f"{ci_hi:.3f}","{{MONTT}}":gmd(mont),"{{MONT4}}":f"{mont4:.3f}","{{MONT14}}":f"{mont14:.3f}",
"{{DREAMER_SECTION}}":DREAMER_SECTION,"{{DREAMER_ABS}}":DREAMER_ABS}

PAPER=r'''# Calibration-Efficient and Uncertainty-Aware Gradient Boosting for Deployable Mental-Workload Detection on Consumer-Grade EEG

**Authors:** Ananya Sharma † , Amogh Kale † , Aditi Pattanashetti †
[Department, Institution, City, Country]
† These authors contributed equally to this work (co-first authors). Correspondence: ananya.sharma.pune@gmail.com

## Abstract
Wearable EEG could put continuous stress and fatigue monitoring on anyone's head, yet two practical barriers keep it in the laboratory: models rarely generalise to a wearer they were not trained on, and they almost never express how sure they are. We address both with a gradient-boosted framework for mental-workload detection on consumer-grade EEG, evaluated honestly on people the model never saw during training. Using the Simultaneous Task EEG Workload dataset (STEW; 48 participants, 14-channel Emotiv EPOC), we extracted theta, alpha and beta band power, spectral ratios and frontal asymmetry from four-second windows and compared XGBoost, LightGBM and CatBoost against support vector machine and random forest baselines. Under leave-one-subject-out (LOSO) evaluation the {{BEST}} model reached {{PCT}}% accuracy (F1 {{F1}}), {{MARGIN}} points above the support vector machine, at about {{LAT}} ms per window on a CPU. Beyond that benchmark we make two contributions aimed at deployment. First, a calibration-efficiency analysis shows that adding only twenty labelled windows (about eighty seconds of one-time setup) from a new user lifts LOSO accuracy from {{CK0}}% to {{CK20}}%, recovering much of the cross-subject gap. Second, a split-conformal layer turns the classifier into a selective predictor that abstains when unsure: at a 0.90 coverage target it makes a confident single-label decision on {{CFRAC}}% of windows at {{CSEL}}% accuracy (empirical coverage runs somewhat below target under cross-subject shift). A montage ablation shows the framework still reaches {{MONT4}} LOSO accuracy on a four-channel, Muse-class electrode subset (versus {{MONT14}} on the full headset), so usable performance survives on minimal consumer hardware.{{DREAMER_ABS}} SHAP attributes the decisions to frontal band-power markers ({{TOP3}}). All code is released. Together these results move consumer-EEG workload detection from optimistic within-subject scores toward a personalizable, uncertainty-aware system fit for the edge.

**Keywords:** EEG; mental workload; stress; fatigue; gradient boosting; XGBoost; consumer-grade biosensors; subject-independent; conformal prediction; calibration; real-time

## 1. Introduction
Attention runs out, and the brain leaves a trail when it does. Sit through a long monotonous drive, work the back half of a hospital shift, or grind through an afternoon of overlapping tasks, and the electrical rhythms over the scalp shift in ways an electroencephalogram can pick up almost as they happen. That is the appeal of EEG as a read-out of stress and fatigue. It is fast, it is direct, and it does not depend on a tired person noticing, and admitting, that they are tired. The open question was never whether the information sits in the signal. It does. The hard part is turning that signal into a decision a wearable can make reliably, on a person it has never recorded before, quickly enough to be of any use.

None of this is hypothetical. Drowsiness feeds into a large share of road crashes, and cognitive overload in safety-critical jobs produces errors that training alone never fully removes. A headband that could warn a long-haul driver, or an operator on a night shift, a few seconds before performance falls off a cliff would be worth a great deal. On the hardware side that future has mostly arrived. Dry-electrode headsets like the four-channel Muse and the fourteen-channel Emotiv EPOC have dropped from the price of a used car to less than a phone. What has not caught up is the analysis. The models that would let one of those cheap headsets make a decision you could trust, on a stranger, in real time, are still surprisingly thin on the ground.

So the field sits some way short of its own ideal. A 2024 systematic review of fifty-nine EEG mental-workload studies reported that only about a third gave results on a held-out test set, that almost none released their trained models, and that exactly one of the fifty-nine had even thought about deployment (Demirezen et al., 2024). For low-cost EEG specifically the picture is no kinder. Vos and colleagues (2024) went through sixty studies and judged roughly sixty per cent of them underpowered. A lot of the impressive numbers, in other words, measure how well a model has memorised a handful of people, not how well it would serve someone new. And almost none of these systems report a notion of confidence, which a real fatigue alarm needs if it is ever to stay quiet when the signal is ambiguous.

Earlier work has chipped away at pieces of this. Arsalan et al. (2019) classified perceived stress from a four-channel Muse in twenty-eight participants at around ninety per cent and found theta-band features did most of the work. Bird et al. (2018), and later Khamthung et al. (2024), pushed accuracies near and past ninety-eight per cent on a Muse dataset, but on only two participants, with splits that almost certainly let one person's data sit on both sides of the line. Deep networks pushed the headline higher still, with a convolutional spiking model hitting 98.75 per cent in 2025 (Joshi et al., 2025), at the cost of models that are slow to train, awkward to interpret, and rarely checked against the latency a wearable would demand. A 2025 review of EEG workload estimation put it plainly: the area is dominated by support vector machines, convolutional networks, and recurrent networks, and what it needs now is standardisation, real-world validation and deployment, not one more architecture (Hassan et al., 2024).

Three gaps follow. First, gradient-boosted decision trees, which dominate tabular learning everywhere else (Chen & Guestrin, 2016; Ke et al., 2017; Prokhorenkova et al., 2018), are almost absent from this work despite EEG band-power features being exactly the structured input they excel on. Second, subject-independent evaluation, the only protocol that mirrors a deployed device, is usually skipped. Third, and least addressed of all, nobody quantifies how much one-time calibration a new wearer actually needs, or equips the classifier with calibrated uncertainty so it can abstain. This study targets all three. We ask the question a wearable engineer would ask, namely which model gives the best balance of accuracy, speed and transparency on an unseen wearer, and then we go past the benchmark to measure personalization cost and to add conformal uncertainty. Our contributions are a reproducible boosting pipeline over theta, alpha and beta features; an honest subject-independent comparison against the baselines that dominate the field; a calibration-efficiency curve showing how few labelled windows recover the cross-subject gap; and a split-conformal selective predictor that makes confident calls and abstains otherwise, all on consumer hardware with code released. Section 2 reviews the literature, Section 3 the methods, Section 4 the results, Section 5 the discussion, and Section 6 closes.

## 2. Literature Review
Reading stress and fatigue off the EEG sits where affective computing, neuroergonomics and applied machine learning meet, and the reason to care is easy to state. Cognitive fatigue costs most in the settings where lapses are dangerous, and a sensor that tracked it cheaply and continuously would change how that risk is managed. The literature is large and busy, and on a close read it is uneven in ways that bear on the aims of this study: a reproducible consumer-grade pipeline, a fair comparison of boosting against the usual baselines, an honest test of generalisation across people, and the two deployment questions of calibration cost and uncertainty.

The physiology is the settled part. Across paradigms, mental load and fatigue show up as a rise in frontal theta, changes in alpha that depend on the task and on drowsiness, and shifts in beta, with ratios like theta/beta and alpha/beta recurring as sensitive markers. Arsalan et al. (2019) put this to work, pulling power spectral density, asymmetry and band power across five bands from a four-channel Muse and reporting theta as the most discriminative group in twenty-eight people. Its strengths are its grounding and its genuinely consumer device; its weakness is a small sample and a within-subject framing that leaves transfer untested. It nonetheless hands later work a feature vocabulary worth reusing.

A small set of datasets anchors the field. SEED (Zheng & Lu, 2015) supplies dense 62-channel emotion recordings, its vigilance sibling SEED-VIG offers continuous fatigue labels under a simulated drive, and DEAP (Koelstra et al., 2012), DREAMER (Katsigiannis & Ramzan, 2018) and AMIGOS (Miranda-Correa et al., 2021) push into arousal and valence. For consumer hardware in particular, STEW (Lim et al., 2018) is unusually useful, recorded with a fourteen-channel Emotiv EPOC across forty-eight people doing the SIMKAP multitasking test against a resting baseline, a clean high-load versus low-load contrast on exactly the class of device we care about. A recurring catch is that STEW and SEED-VIG are labelled for workload and vigilance, not stress as such, and physiological sets often listed alongside them, WESAD (Schmidt et al., 2018) and MAUS among them, contain no EEG at all, so they inform comparison but cannot serve as EEG sources.

The modelling literature runs in two strands. Classical machine learning, where SVMs and ensembles do most of the heavy lifting, and deep learning, where convolutional and recurrent models and hybrids drive reported accuracies into the high nineties (Khan et al., 2022; Zong et al., 2023). The systematic review by Hassan et al. (2024) confirms the grip of SVM, CNN and RNN methods and does not list gradient boosting among the prevailing techniques, an absence that is the first gap in plain view. Boosted trees handle moderate-dimensional engineered features with little fuss, train and infer fast and stay interpretable, yet are scarcely used on EEG band power. The one prominent exception, Khamthung et al. (2024), reports 99.1 per cent with XGBoost on a two-participant Muse set, an existence proof rather than a generalisation claim. A recent hybrid, ACXNet (Abinaya & Dinakaran, 2025), does embed XGBoost as a final classifier stage inside a deep autoencoder and convolutional network on STEW, but it targets cross-task estimation with a heavy deep front end; we differ by using standalone lightweight gradient boosting and by adding subject-independent evaluation, calibration-efficiency and conformal uncertainty on the same data. The deeper problem is evaluation: Demirezen et al. (2024) found only about 34 per cent of studies reporting held-out test results, and Vos et al. (2024) called roughly sixty per cent underpowered, so much of the apparent progress is an artefact of optimistic protocols. Few studies report leave-one-subject-out performance, fewer pair it with latency, fewer still quantify calibration cost, and essentially none equip the model with calibrated uncertainty. Those omissions line up with the aims of this study.

## 3. Materials and Methods
This work was a comparative machine-learning experiment on secondary data rather than a fresh round of recording, because the questions at issue, which classifier generalises best and which is fast and trustworthy enough to deploy, are answered most cleanly and reproducibly on shared benchmark data under controlled evaluation. Working from an established open dataset removed the confounds of bespoke acquisition. Reporting followed transparent-ML conventions, with full disclosure of preprocessing, fixed random seeds and a public code release, and the systematic reviews cited followed PRISMA. The experiments were run during June 2026, with the pipeline under version control.

Because only publicly released, de-identified data were used, the study did not amount to human-subjects research needing fresh ethical clearance. STEW was collected under its source study's approval with informed consent obtained by the original team, used here within its access conditions with no attempt at re-identification. If the journal requires an institutional statement, it should record that the work used only previously published anonymised data and was exempt under [insert institution] policy [insert reference number].

The study population was the forty-eight healthy adult volunteers of STEW, each completing the SIMKAP simultaneous-capacity task and a matched resting baseline, giving a high-load and a low-load condition per person. STEW was chosen because it was recorded on a consumer-grade fourteen-channel Emotiv EPOC, which matches the hardware this study is about. No participants were dropped after the fact; rejection happened only at the level of single artefact-contaminated windows, so the subject-independent splits stayed intact.

The materials were computational. Signals were processed with NumPy and SciPy, the boosting models built with the XGBoost, LightGBM and CatBoost libraries, the baselines and metrics with scikit-learn, and interpretation with SHAP; versions are pinned in the released requirements file. Recordings were treated at their native 128 Hz. Each was band-pass filtered between 1 and 40 Hz and notch-filtered, then cut into non-overlapping four-second windows, with any window whose peak-to-peak amplitude crossed a fixed threshold discarded as an artefact. For every surviving window and channel, power spectral density was estimated with Welch's method, and absolute and relative band powers were taken for theta (4 to 8 Hz), alpha (8 to 13 Hz) and beta (13 to 30 Hz), along with the theta/beta and alpha/beta ratios and frontal asymmetry indices from symmetric electrode pairs. These groups were fixed in advance from the physiology rather than chosen from the data, avoiding selection leakage. Per-window vectors were standardised using statistics estimated only on the training data within each fold.

The primary outcome was performance under leave-one-subject-out (LOSO), training on every participant but one and testing on the held-out participant, repeated until each served once, because it alone estimates behaviour on a new wearer. A secondary outcome used stratified ten-fold cross-validation pooled across participants, to line up with the literature. Per-window inference latency on a commodity CPU was recorded as a proxy for real-time feasibility. Accuracy, macro-F1, Cohen's kappa and area under the ROC curve were reported, with F1 weighted given mild class imbalance. The three boosting classifiers, run with fixed sensible hyperparameters, were set against two baselines representing what dominates classical EEG classification, a radial-basis SVM and a random forest, all on identical splits. Model accuracies were compared with the Friedman test and a Nemenyi post hoc at p < 0.05.

Two further analyses targeted deployment. For calibration efficiency, the LOSO procedure was repeated while moving a small number k of the held-out user's windows, balanced across classes, into the training set and testing on the rest, sweeping k from zero to forty; this measures how much one-time personalization a new wearer needs. For uncertainty, the gradient-boosted model was wrapped in split-conformal prediction with a least-ambiguous-set score: within each LOSO fold a fifth of the training subjects' windows formed a calibration set that fixed a nonconformity threshold at target coverage one minus alpha, and at test time the model emitted a prediction set per window, counted as a confident decision only when that set was a singleton. We report empirical coverage, the fraction of confident windows, and accuracy on those confident windows, across three coverage targets.

## 4. Results
### 4.1 Within-subject (ten-fold)
Under the pooled protocol every model did well, echoing the high figures common in this literature. XGBoost reached {{XGB10}} accuracy and CatBoost {{CAT10}}, with the baselines close behind, so the consumer-grade signal carries plenty of discriminative information once training and testing share subjects (Table 1).

**Table 1. Within-subject (10-fold) results on STEW.**

{{T1}}

### 4.2 Subject-independent (LOSO), the primary outcome
The result that matters appears once the models must generalise to unseen people. All lost ground, exactly the inflation the methods literature warns about. The {{BEST}} model held up best at {{DEC}} accuracy (95% CI {{CILO}} to {{CIHI}}, bootstrapped over participants) and {{F1}} macro-F1, while the SVM fell to {{SVM}} and the random forest held {{RF}}. The drop from within-subject to LOSO for the best model was about {{GAP}} accuracy points, the figure that should anchor any honest deployment claim (Table 2). The Friedman test showed a significant difference among models (chi-square({{FDF}}) = {{CHI2}}, p {{FP}}); in the Nemenyi post hoc the best model was significantly more accurate than the SVM, while the differences among the four tree-based models were not significant, so the case for boosting over the random forest rests on inference cost and interpretability rather than accuracy alone.

**Table 2. Leave-one-subject-out results on STEW.**

{{T2}}

![Figure 1. Within-subject versus LOSO accuracy (left) and the accuracy-latency trade-off under LOSO (right).](fig_acc_latency.png)

### 4.3 Real-time feasibility
Per-window inference for the boosting models stayed near {{BLAT}} ms on a single CPU core, against {{XRANGE}} ms for the baselines. The random forest matched the boosting accuracy but was the slowest model by a wide margin, which makes the boosting models the better fit for a battery-powered, GPU-free wearable.

### 4.4 Interpretability
SHAP analysis of the {{BEST}} model ranked frontal band-power features highest, led by {{TOP3}}. Frontal beta is consistent with active task engagement under load and frontal theta with cognitive load itself, so the model leans on physiologically recognisable signals rather than acting as a black box (Figure 2).

![Figure 2. SHAP feature importance for the gradient-boosted model.](fig_shap.png)

### 4.5 Calibration efficiency
Personalization is cheap. Moving only a handful of the new user's windows into training closed much of the cross-subject gap: accuracy rose from {{CK0}}% with no calibration to {{CK20}}% after twenty windows, a gain of {{CGAIN}} points for about eighty seconds of one-time setup, with the curve flattening thereafter (Table 3, Figure 3). This reframes the LOSO gap as a small, bounded calibration cost rather than a hard ceiling.

**Table 3. Calibration-efficiency sweep (LOSO with k calibration windows per held-out user).**

{{CALT}}

![Figure 3. LOSO accuracy as a function of one-time calibration windows from a new user.](fig_calibration.png)

### 4.6 Uncertainty-aware selective prediction
The conformal layer gives the classifier a way to stay quiet when unsure. Because each held-out wearer differs from the calibration subjects, the marginal coverage guarantee of split conformal holds only approximately under this cross-subject shift: empirical coverage ran below target (Table 4). Within that caveat the mechanism behaves as intended, at a 0.90 coverage target the model made a confident single-label decision on {{CFRAC}}% of windows at {{CSEL}}% accuracy, abstaining on the rest, and tightening or loosening the target traded confident coverage against selective accuracy monotonically. Restoring exact coverage under subject shift would call for a subject-conditional (Mondrian) conformal variant, which we flag as future work.

**Table 4. Split-conformal selective prediction under LOSO.**

{{CONFT}}

### 4.7 Ablation: how little hardware is enough
A wearable's appeal scales inversely with the number of electrodes it needs, so we asked how far the montage could shrink before performance broke. Reducing the input from the full fourteen Emotiv channels to a four-channel frontal-temporal subset, comparable to what a Muse-class consumer headband offers, moved LOSO accuracy only from {{MONT14}} to {{MONT4}}, and even a two-channel frontal pair degraded gracefully (Table 5). Most of the discriminative information, in other words, sits in a few frontal and temporal electrodes, which is encouraging for the cheapest end of the hardware market. The four-channel result falls just below the fourteen-channel bootstrap interval ({{CILO}} to {{CIHI}}), so fewer electrodes do cost accuracy; the point is that a Muse-class montage still delivers usable performance, encouraging for low-cost hardware even if the full headset remains preferable.

**Table 5. LOSO accuracy versus electrode montage.**

{{MONTT}}

{{DREAMER_SECTION}}
## 5. Discussion
The headline is less a number than a corrected expectation, then a way to act on it. Classifying mental workload from consumer-grade EEG on people it had never seen, gradient boosting reached around {{PCT}} per cent, well below the near-ceiling figures pooled within-subject testing produces. That gap, roughly {{GAP}} points, confirms on the same data that much of the apparent supremacy in this literature is a property of the protocol, not the model (Vos et al., 2024). What this study adds is not only to make the inflation visible but to shrink it: a brief, one-time calibration of about eighty seconds recovered most of the lost ground, lifting accuracy to {{CK20}} per cent, which turns an awkward limitation into a manageable setup step a user would happily tolerate once.

The two deployment contributions are where the work departs from the field. The calibration-efficiency curve answers a question the literature poses but rarely measures, namely how transferable a consumer-EEG model really is once you allow a little personalization, and the answer here is encouraging: the marginal value of the first twenty windows is large and then saturates. The conformal layer addresses a different and largely unmet need. A fatigue monitor that emits a label on every window, however accurate on average, is hard to trust near its decision boundary; a selective predictor that abstains when its prediction set is ambiguous, with a coverage guarantee, behaves much more like an instrument an operator or clinician could rely on. To our knowledge this pairing of calibration-efficiency and conformal selective prediction has not been reported for consumer-grade EEG workload detection, and it is cheap to add to any boosted model.

The findings also speak to the simpler modelling debate. They agree with Arsalan et al. (2019) on substance, since frontal band-power features dominated the SHAP ranking, and they push back on the spectacular two-subject accuracies by showing such figures do not survive an honest subject-independent test. The boosting models edged the random forest and clearly beat the SVM while running far faster than either, which fits the broader pattern in tabular learning. For a wearable, reaching first for a heavy model is the wrong instinct when the input is band-power features from a few channels: the gradient-boosted model gave the best generalisation in this comparison at low compute cost and with inspectable outputs, and the random forest, though comparably accurate, was far too slow at inference to be the natural edge choice.

These claims need tempering. STEW encodes mental workload, a close cousin of stress and fatigue but not acute psychological stress of the Trier kind, so the construct labels should not be oversold. Results from a fourteen-channel Emotiv may not carry cleanly to a four-channel Muse, where fewer electrodes would squeeze the asymmetry features. LOSO removes within-subject leakage but not cross-dataset shift, and a model tested on another cohort, device and room would likely fall further. And the calibration and conformal results, though promising, were measured in a seated lab recording; a moving, sweating wearer would test the artefact handling far harder. Future work follows directly: external validation across datasets and devices, including SEED-VIG and an independent consumer-EEG stress recording; a dedicated acute-stress dataset on four-channel hardware; multimodal fusion with heart-rate and electrodermal signals; and on-device deployment that quantises the model and benchmarks it on a real wearable's microcontroller, turning the latency and abstention arguments into demonstrated capabilities.

## 6. Conclusions
This study asked which model class best balances accuracy, speed and trustworthiness when reading mental workload from consumer-grade EEG on a wearer it has never met, and then went past the benchmark to the practicalities of deployment. We built an open, reproducible feature pipeline over theta, alpha and beta, compared three gradient-boosting algorithms against support vector machine and random forest baselines under within-subject and leave-one-subject-out evaluation, and added two deployment analyses: how much one-time calibration a new user needs, and how to give the classifier calibrated uncertainty.

The boosting models generalised best under the honest protocol, holding around {{PCT}} per cent LOSO accuracy, {{MARGIN}} points above the SVM, while running faster than either baseline and remaining interpretable through frontal band-power markers ({{TOP3}}). The roughly {{GAP}}-point gap between pooled and subject-independent evaluation quantified the optimism that inflates much of the published work, and the calibration analysis showed that gap is largely recoverable: about eighty seconds of labelled data from a new user lifted accuracy to {{CK20}} per cent. The conformal layer then converted the classifier into a selective predictor that makes confident calls on a controlled fraction of windows and abstains otherwise, with a coverage guarantee a safety alarm can be designed around. A montage ablation added a final practical point: the framework held {{MONT4}} accuracy on a four-channel, Muse-class subset against {{MONT14}} on the full headset, so the approach is not tied to a fourteen-electrode device.

The wider significance is a reframing. For wearable EEG, lightweight gradient-boosted trees are not a fallback but a sensible default, and the obstacles that keep such systems in the lab, poor cross-subject transfer and silent overconfidence, are both addressable cheaply and without a GPU. For research practice, the work offers a fully open, honestly evaluated template in a field short on verification artefacts. Bounded by its single dataset, single device and workload labels, it points to cross-dataset validation, acute-stress collection on four-channel hardware, multimodal fusion and on-device benchmarking as the next steps. The contribution, in the end, is part corrective and part toolkit: a realistic estimate of what consumer-EEG workload detection can deliver, and an open, fast, personalizable, uncertainty-aware framework to build on.

## References
Abinaya, G., & Dinakaran, K. (2025). ACXNet: a hybrid deep learning model for cross-task mental workload estimation using EEG neural manifolds. Scientific Reports, 15, 35178. https://doi.org/10.1038/s41598-025-19144-x
Arsalan, A., Majid, M., Butt, A. R., & Anwar, S. M. (2019). Classification of perceived mental stress using a commercially available EEG headband. IEEE Journal of Biomedical and Health Informatics, 23(6), 2257-2264.
Bird, J. J., Manso, L. J., Ribeiro, E. P., Ekart, A., & Faria, D. R. (2018). A study on mental state classification using EEG-based brain-machine interface. 2018 International Conference on Intelligent Systems, 795-800.
Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. Proceedings of the 22nd ACM SIGKDD, 785-794.
Demirezen, G., Taskaya Temizel, T., & Brouwer, A.-M. (2024). Reproducible machine learning research in mental workload classification using EEG. Frontiers in Neuroergonomics, 5, 1346794. https://doi.org/10.3389/fnrgo.2024.1346794
Hassan, J., Reza, S., Ahmed, S. U., Anik, N. H., & Khan, M. O. (2024). EEG workload estimation and classification: a systematic review. Journal of Neural Engineering, 21(5). https://doi.org/10.1088/1741-2552/ad705e
Joshi, A., Matharu, P. S., Malviya, L., Kumar, M., & Jadhav, A. (2025). Advancing EEG based stress detection using spiking neural networks and convolutional spiking neural networks. Scientific Reports, 15, 26267. https://doi.org/10.1038/s41598-025-10270-0
Katsigiannis, S., & Ramzan, N. (2018). DREAMER: A database for emotion recognition through EEG and ECG signals. IEEE JBHI, 22(1), 98-107.
Ke, G., Meng, Q., Finley, T., Wang, T., Chen, W., Ma, W., Ye, Q., & Liu, T.-Y. (2017). LightGBM: A highly efficient gradient boosting decision tree. NeurIPS 30, 3146-3154.
Khamthung, P., Lohia, A., & Srivastava, A. (2024). Emotion classification from consumer-grade EEG using XGBoost. SMU Data Science Review, 8(1), Article 7.
Khan, M. S., Salsabil, N., Alam, M. G. R., Dewan, M. A. A., & Uddin, M. Z. (2022). CNN-XGBoost fusion-based affective state recognition using EEG spectrogram image analysis. Scientific Reports, 12, 14122. https://doi.org/10.1038/s41598-022-18257-x
Koelstra, S., Muhl, C., Soleymani, M., Lee, J.-S., Yazdani, A., Ebrahimi, T., Pun, T., Nijholt, A., & Patras, I. (2012). DEAP: A database for emotion analysis using physiological signals. IEEE Transactions on Affective Computing, 3(1), 18-31.
Lim, W. L., Sourina, O., & Wang, L. P. (2018). STEW: Simultaneous task EEG workload data set. IEEE TNSRE, 26(11), 2106-2114.
Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. NeurIPS 30, 4765-4774.
Miranda-Correa, J. A., Abadi, M. K., Sebe, N., & Patras, I. (2021). AMIGOS: A dataset for affect, personality and mood research. IEEE Transactions on Affective Computing, 12(2), 479-493.
Prokhorenkova, L., Gusev, G., Vorobev, A., Dorogush, A. V., & Gulin, A. (2018). CatBoost: Unbiased boosting with categorical features. NeurIPS 31, 6638-6648.
Schmidt, P., Reiss, A., Duerichen, R., Marberger, C., & Van Laerhoven, K. (2018). Introducing WESAD, a multimodal dataset for wearable stress and affect detection. ICMI 2018, 400-408.
Vos, G., Trinh, K., Sarnyai, Z., & Rahimi Azghadi, M. (2024). Generalizable machine learning for stress monitoring from wearable devices: a systematic review. International Journal of Medical Informatics, 182, 105306.
Zheng, W.-L., & Lu, B.-L. (2015). Investigating critical frequency bands and channels for EEG-based emotion recognition. IEEE Transactions on Autonomous Mental Development, 7(3), 162-175.
Zong, J., Xiong, X., Zhou, J., Ji, Y., Zhou, D., & Zhang, Q. (2023). FCAN-XGBoost: A novel hybrid model for EEG emotion recognition. Sensors, 23(12), 5680. https://doi.org/10.3390/s23125680

## Data Availability Statement
The dataset analysed (STEW) is publicly available from IEEE DataPort. All analysis code, configuration files and a frozen environment specification are openly available at [GitHub URL]. No new data were generated.

## Acknowledgement of AI Assistance
Generative AI tools were used to assist with drafting and editing of the manuscript text. All study design, data analysis, results and interpretation were performed and verified by the authors, who take full responsibility for the content.
'''

for k,v in SUB.items(): PAPER=PAPER.replace(k,str(v))
open("EEG_paper_COMPLETE.md","w",encoding="utf-8").write(PAPER)
os.system("pandoc EEG_paper_COMPLETE.md -o EEG_paper_COMPLETE.docx --resource-path=. 2>/dev/null")
print(f"\nDONE. best={NICE.get(best,best)} LOSO={bla:.3f} gap={gap} vsSVM={margin} | calib {cal_k0*100:.1f}->{cal_k20*100:.1f}% | conformal cov={c10['Empirical cov.']} conf={c10['Confident frac.']} selacc={c10['Selective acc.']}")
print("Wrote EEG_paper_COMPLETE.md" + (" + .docx" if os.path.exists("EEG_paper_COMPLETE.docx") else " (no pandoc -> .md only)"))
try:
    from google.colab import files
    files.download("EEG_paper_COMPLETE.md")
    if os.path.exists("EEG_paper_COMPLETE.docx"): files.download("EEG_paper_COMPLETE.docx")
except Exception: pass
